In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

inputs = tokenizer("Hello world", return_tensors="pt")
output = model(**inputs, use_cache=True)

print(type(output.past_key_values))
print(len(output.past_key_values))

c:\research-eng\repro-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5124.83it/s]


<class 'transformers.cache_utils.DynamicCache'>
24


In [4]:
print(dir(output.past_key_values))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'batch_repeat_interleave', 'batch_select_indices', 'batch_size', 'crop', 'early_initialization', 'get_mask_sizes', 'get_max_cache_shape', 'get_max_length', 'get_seq_length', 'has_previous_state', 'is_compileable', 'is_initialized', 'is_linear', 'is_sliding', 'layer_class_to_replicate', 'layers', 'max_batch_size', 'max_cache_len', 'offload', 'offloading', 'prefetch', 'reorder_cache', 'reset', 'update', 'update_conv_state', 'update_indexer', 'update_recurrent_state']


In [5]:
first_layer = output.past_key_values.layers[0]
print(type(first_layer))
print(dir(first_layer))

<class 'transformers.cache_utils.DynamicLayer'>
['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', 'batch_repeat_interleave', 'batch_select_indices', 'crop', 'device', 'dtype', 'get_mask_sizes', 'get_max_cache_shape', 'get_max_length', 'get_seq_length', 'is_compileable', 'is_initialized', 'is_sliding', 'keys', 'layer_type', 'lazy_initialization', 'offload', 'prefetch', 'reorder_cache', 'reset', 'supports_early_init', 'update', 'values']


In [6]:
first_layer_key = output.past_key_values.layers[0].keys
first_layer_value = output.past_key_values.layers[0].values
print(first_layer_key.shape)
print(first_layer_value.shape)

torch.Size([1, 2, 2, 64])
torch.Size([1, 2, 2, 64])


Let's decode that shape against (batch, heads, seq_len, head_dim):

1 → batch size (you ran one sentence, so this checks out)
2 → this should be num_heads... but wait, does that match what you'd expect for a 0.5B model? Models this size sometimes use grouped-query attention (GQA), where the number of key/value heads is smaller than the number of query heads. So this 2 might be num_kv_heads, not the full num_heads you'd use for queries. Worth checking — do you know if Qwen2.5-0.5B uses GQA, or does it use the same head count for Q and K/V?
2 → sequence length — does this match how many tokens "Hello world" actually tokenized into? Can you check with tokenizer("Hello world") or inputs["input_ids"].shape to confirm?
64 → head_dim — does 64 line up with what you'd calculate as embed_dim // num_heads for this model (if you look up Qwen2.5-0.5B's config)?

In [7]:
model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2